<a href="https://colab.research.google.com/github/luzangelacarabali/Python_Cursos/blob/main/normalizacion_tests_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Normalización de archivos — Oriente Estéreo
**Python Schools 2026**

Convierte nombres de archivos de audio al formato estándar:
```
titulo_AAAA-MM-DD.ext
titulo_AAAA-MM-DD_epNNN.ext   (si tiene episodio)
titulo_sin_fecha.ext           (si no tiene fecha)
```

## 1. Carga de librerías

In [ ]:
from pathlib import Path

## 2. Carga de datos — Escanear la carpeta

Aquí se leen los archivos reales de la carpeta.
El escáner busca `.mp3` y `.mp4` y devuelve la lista de nombres.

In [ ]:
carpetas = [
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE",
]

In [ ]:
archivos_prueba = [
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 06-06-25.mp3",
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 17-07-25.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES- AGOSTO 25 DE 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES DIC 1 DE 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/En la Raya Deportes- Septiembre 29 de 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DEJAME SER ARTE 03-07-25.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DSA 02-10-25.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA/Arcangel - Andan Diciendo.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA/GAMBOA - Fuiste Tu.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/CABEZOTE DSA.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/PISADOR DSA 1.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/SEPARADOR DSA 3 FINAL.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/FONDO DSA.jpg",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/FONDO DSA",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 35 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 118.mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 122 2023 (1).mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 290 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - mujeres bolero parte 2.mp3",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - boleros inspirados estaciones dima.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/1 - La historia de la radio y la radio en la historia.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/7 - Comienzos de la radio en Colombia.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/13 - Radio educativa, Radio Sutatenza.mp3",
]

## 3. Definición de funciones

### 3.1 Generación automática del diccionario de meses

In [ ]:
# En lugar de escribir cada mes a mano, definimos una lista base
# con el nombre largo y la abreviación más usada.
# El loop de abajo genera el diccionario solo.

NOMBRES_MESES = [
    ("enero",       "ene"),
    ("febrero",     "feb"),
    ("marzo",       "mar"),
    ("abril",       "abr"),
    ("mayo",        "may" ),
    ("junio",       "jun"),
    ("julio",       "jul"),
    ("agosto",      "ago"),
    ("septiembre",  "sep"),
    ("octubre",     "oct"),
    ("noviembre",   "nov"),
    ("diciembre",   "dic"),
]

# Construir el diccionario automáticamente
MESES = {}
for numero, (nombre_largo, abrev) in enumerate(NOMBRES_MESES, start=1):
    num_str = str(numero).zfill(2)   # 1 -> '01', 9 -> '09'
    MESES[nombre_largo] = num_str     # 'agosto'     -> '08'
    if abrev:
        MESES[abrev] = num_str        # 'ago'        -> '08'
    MESES[nombre_largo[:4]] = num_str # 'agos'       -> '08' (primeras 4 letras)
    MESES[nombre_largo[:3]] = num_str # 'ago' ya existe, pero cubre 'sep','oct'...

# Solo los nombres largos sirven para fuzzy matching (evitar falsos positivos)
MESES_LARGOS = [nombre for nombre, _ in NOMBRES_MESES]

# Palabras que no son parte del título y se eliminan al normalizar
PALABRAS_RUIDO = {
    "de", "del", "la", "el",
    "sabado", "domingo", "lunes", "martes", "miercoles", "jueves", "viernes"
}

In [ ]:
# Ver el resultado
print("Meses detectables:", list(MESES.keys()))

Meses detectables: ['enero', 'ene', 'ener', 'febrero', 'feb', 'febr', 'marzo', 'mar', 'marz', 'abril', 'abr', 'abri', 'mayo', 'may', 'junio', 'jun', 'juni', 'julio', 'jul', 'juli', 'agosto', 'ago', 'agos', 'septiembre', 'sep', 'sept', 'octubre', 'oct', 'octu', 'noviembre', 'nov', 'novi', 'diciembre', 'dic', 'dici']


In [ ]:
print("Palabras de ruido:", PALABRAS_RUIDO)

Palabras de ruido: {'la', 'el', 'lunes', 'domingo', 'miercoles', 'viernes', 'de', 'jueves', 'sabado', 'martes', 'del'}


### 3.2 Funciones de apoyo

In [ ]:
def separar_extension(nombre):
    """Separa el nombre base de la extensión.
    Ejemplo: 'audio.MP3' -> ('audio', '.mp3')
    """
    if "." in nombre:
        base, ext = nombre.rsplit(".", 1)
        return base, "." + ext.lower()
    return nombre, ""


def limpiar_texto(texto):
    """Pasa el texto a minúsculas y reemplaza
    caracteres especiales por espacio.
    """
    texto = texto.lower()
    for caracter in [",", ".", "(", ")", "!", ";"]:
        texto = texto.replace(caracter, " ")
    return texto


def expandir_anio(anio_str):
    """Convierte un año de 2 dígitos a 4 dígitos.
    Ejemplo: '25' -> '2025', '2025' -> '2025'
    Retorna None si el valor no es un año válido.
    """
    if not anio_str.isdigit():
        return None
    if len(anio_str) == 2:
        return "20" + anio_str
    if len(anio_str) == 4 and 1900 <= int(anio_str) <= 2100:
        return anio_str
    return None


def construir_fecha(dia, mes, anio_str):
    """Arma la fecha en formato AAAA-MM-DD.
    Si día y mes están al revés, los corrige automáticamente.
    Retorna None si los valores son inválidos.
    """
    anio = expandir_anio(anio_str)
    if not anio:
        return None

    d, m = int(dia), int(mes)

    # Verificar si día y mes son válidos
    if not (1 <= m <= 12 and 1 <= d <= 31):
        # Intentar intercambiarlos
        if 1 <= d <= 12 and 1 <= m <= 31:
            d, m = m, d
        else:
            return None

    return f"{anio}-{str(m).zfill(2)}-{str(d).zfill(2)}"


def buscar_mes(token):
    # Solo coincidencia exacta (sin difflib)
    return MESES.get(token)

### 3.3 Función principal: `detectar_fecha`

In [ ]:
def detectar_fecha(nombre):
    """Detecta la fecha en el nombre de un archivo.

    Soporta estos formatos (en orden de prioridad):
      - AAAA-MM-DD  (ISO)          Archivo 2025-08-15.mp3
      - DD-MM-AA(AA)               BARRIO ADENTRO 06-06-25.mp3
      - DD/MM/AAAA                 Noticias 06/11/2025.mp3
      - DD.MM.AAAA                 Programa 14.03.2026.mp3
      - Mes en español + día + año AGOSTO 25 DE 2025
      - Solo año (último recurso)  Episodio 35 2025.mp3

    Retorna: 'AAAA-MM-DD' | 'AAAA-MM' | 'AAAA' | 'sin_fecha'
    """
    base, _ = separar_extension(nombre)
    texto = limpiar_texto(base)
    palabras = texto.split()

    # 1. Formato ISO: AAAA-MM-DD (una palabra con dos guiones)
    for pal in palabras:
        if pal.count("-") == 2:
            partes = pal.split("-")
            if len(partes) == 3 and partes[0].isdigit() and partes[1].isdigit() and partes[2].isdigit():
                anio, mes, dia = partes
                if len(anio) == 4 and 1 <= int(mes) <= 12 and 1 <= int(dia) <= 31:
                    fecha = construir_fecha(dia, mes, anio)
                    if fecha:
                        return fecha

    # 2. DD-MM-AAAA o DD-MM-AA
    for pal in palabras:
        if pal.count("-") == 2:
            partes = pal.split("-")
            if len(partes) == 3 and partes[0].isdigit() and partes[1].isdigit() and partes[2].isdigit():
                dia, mes, anio = partes
                if 1 <= int(dia) <= 31 and 1 <= int(mes) <= 12:
                    fecha = construir_fecha(dia, mes, anio)
                    if fecha:
                        return fecha

    # 3. DD/MM/AAAA
    for pal in palabras:
        if pal.count("/") == 2:
            partes = pal.split("/")
            if len(partes) == 3 and partes[0].isdigit() and partes[1].isdigit() and partes[2].isdigit():
                dia, mes, anio = partes
                fecha = construir_fecha(dia, mes, anio)
                if fecha:
                    return fecha

    # 4. DD.MM.AAAA
    for pal in palabras:
        if pal.count(".") == 2:
            partes = pal.split(".")
            if len(partes) == 3 and partes[0].isdigit() and partes[1].isdigit() and partes[2].isdigit():
                dia, mes, anio = partes
                fecha = construir_fecha(dia, mes, anio)
                if fecha:
                    return fecha

    # 5. Mes en español + día + año (ej: "agosto 25 de 2025")
    for i, tok in enumerate(palabras):
        num_mes = buscar_mes(tok)
        if num_mes is None:
            continue
        dia = None
        anio = None
        # Buscar alrededor (±4 palabras)
        for j in range(max(0, i-4), min(len(palabras), i+5)):
            cand = palabras[j]
            if cand.isdigit():
                val = int(cand)
                if 1900 <= val <= 2100:
                    anio = str(val)
                elif 1 <= val <= 31 and j != i and dia is None:
                    dia = str(val).zfill(2)
        if anio and dia:
            fecha = construir_fecha(dia, num_mes, anio)
            if fecha:
                return fecha
        elif anio:
            anio_completo = expandir_anio(anio)
            if anio_completo:
                return f"{anio_completo}-{num_mes}"
        break  # solo el primer mes encontrado

    # 6. Solo año (4 dígitos)
    for pal in palabras:
        if pal.isdigit() and len(pal) == 4 and 1900 <= int(pal) <= 2100:
            return pal

    return "sin_fecha"

### 3.4 Función: `detectar_episodio`

In [ ]:
def detectar_episodio(nombre):
    """Detecta el número de episodio en el nombre de un archivo.

    Reconoce estos estilos:
      - 'Episodio 35 ...'        -> ep035
      - 'PROGRAMA 286 ...'       -> ep286
      - 'PROG. 293 ...'          -> ep293
      - '7 - Título del audio'   -> ep007

    Retorna: 'epNNN' o None si no hay episodio.
    """
    base, _ = separar_extension(nombre)
    texto = limpiar_texto(base)
    palabras = texto.split()

    # Buscar "episodio 35", "programa 293", "prog 293"
    for i, pal in enumerate(palabras):
        if pal in ["episodio", "programa", "prog"]:
            if i + 1 < len(palabras) and palabras[i+1].isdigit():
                num = str(int(palabras[i+1])).zfill(3)
                return "ep" + num

    # Buscar estilo "7 - Título" al principio
    primera_palabra = base.strip().split()[0] if base.strip() else ""
    if "-" in primera_palabra:
        posible_num = primera_palabra.split("-")[0]
        if posible_num.isdigit():
            num = str(int(posible_num)).zfill(3)
            return "ep" + num

    # Alternativa: primer token es número y segundo es "-"
    tokens_base = base.strip().split()
    if len(tokens_base) >= 2 and tokens_base[0].isdigit() and tokens_base[1] == "-":
        num = str(int(tokens_base[0])).zfill(3)
        return "ep" + num

    return None

### 3.5 Función: `normalizar_nombre`

In [ ]:
def limpiar_titulo(nombre, fecha, episodio):
    """Extrae el título limpio: quita la fecha, el episodio
    y palabras de relleno del nombre.
    """
    base, _ = separar_extension(nombre)
    texto = limpiar_texto(base)

    # Eliminar la fecha si existe
    if fecha != "sin_fecha":
        anio = fecha[:4]
        texto = texto.replace(anio, " ")
        if len(fecha) == 10:  # Fecha completa
            mes_num = fecha[5:7]
            dia_num = fecha[8:10]
            texto = texto.replace(mes_num, " ")
            texto = texto.replace(str(int(mes_num)), " ")
            texto = texto.replace(dia_num, " ")
            texto = texto.replace(str(int(dia_num)), " ")
            # Eliminar nombre del mes
            for nombre_mes, num in MESES.items():
                if num == mes_num:
                    texto = texto.replace(nombre_mes, " ")

    # Eliminar número de episodio
    if episodio:
        num_ep = str(int(episodio[2:]))
        texto = texto.replace(num_ep, " ")
        for palabra in ["episodio", "programa", "prog"]:
            texto = texto.replace(palabra, " ")

    # Dividir en palabras, filtrar ruido y limpiar caracteres
    partes = []
    for token in texto.split():
        if token in PALABRAS_RUIDO:
            continue
        if token == "-":
            continue
        if token.isdigit() and 1 <= int(token) <= 31:
            continue
        # Reemplazar cualquier carácter no alfanumérico por '_'
        limpio = ""
        for c in token:
            if c.isalnum():
                limpio += c
            else:
                limpio += "_"
        limpio = limpio.strip("_")
        if limpio:
            partes.append(limpio)

    if not partes:
        return "audio"
    return "_".join(partes)

def normalizar_nombre(nombre):
    _, ext = separar_extension(nombre)
    fecha = detectar_fecha(nombre)
    ep = detectar_episodio(nombre)
    titulo = limpiar_titulo(nombre, fecha, ep)

    if ep:
        return f"{titulo}_{fecha}_{ep}{ext}"
    else:
        return f"{titulo}_{fecha}{ext}"

## 5. Verificación — Probar con tus propios archivos

Escribe en `mis_archivos` los nombres que quieres probar.
Pueden ser nombres reales de tu carpeta o cualquier nombre inventado.
La función detecta la fecha y el episodio **automáticamente** leyendo el nombre.
Los que no tienen fecha aparecen como `sin_fecha`.

In [ ]:
mis_archivos = [
    # Ejemplos: descomenta y añade los tuyos
    # "BARRIO ADENTRO 06-06-25.mp3",
    # "EN LA RAYA AGOSTO 25 DE 2025.mp3",
]

if not mis_archivos:
    mis_archivos = archivos

print(f"{'FECHA':<14} {'EPISODIO':<10} {'NOMBRE'}")
print("-" * 80)
for nombre in mis_archivos:
    fecha = detectar_fecha(nombre)
    episodio = detectar_episodio(nombre) or "sin episodio"
    print(f"{fecha:<14} {episodio:<12} {nombre}")

print("\nResultado normalizado:")
print("-" * 80)
for nombre in mis_archivos:
    print(f"  ANTES  : {nombre}")
    print(f"  DESPUÉS: {normalizar_nombre(nombre)}\n")

FECHA          EPISODIO     NOMBRE
--------------------------------------------------------------------------------
2025-06-06     sin episodio BARRIO ADENTRO 06-06-25.mp3
2025-08-25     sin episodio EN LA RAYA DEPORTES- AGOSTO 25 DE 2025.mp3
2026-03-14     ep293        PROG. 293 ENTRE NOSOTROS, SABADO 14 DE MARZO 2026.mp3
2025           ep035        Episodio 35 2025.mp3
sin_fecha      sin episodio CABEZOTE DSA.mp3

Resultado normalizado:
--------------------------------------------------------------------------------
  ANTES  : BARRIO ADENTRO 06-06-25.mp3
  DESPUES: barrio_adentro_06_06_25_2025-06-06.mp3

  ANTES  : EN LA RAYA DEPORTES- AGOSTO 25 DE 2025.mp3
  DESPUES: en_raya_deportes_2025-08-25.mp3

  ANTES  : PROG. 293 ENTRE NOSOTROS, SABADO 14 DE MARZO 2026.mp3
  DESPUES: prog_entre_nosotros_2026-03-14_ep293.mp3

  ANTES  : Episodio 35 2025.mp3
  DESPUES: episodio_2025_ep035.mp3

  ANTES  : CABEZOTE DSA.mp3
  DESPUES: cabezote_dsa_sin_fecha.mp3



## Crear disco de prueba
Esta funcion crea carpetas y archivos vacios con nombres reales del proyecto para poder probar `normalizar_nombre` sobre archivos reales.

In [ ]:
# Funcion que crea las carpetas y archivos vacios de prueba
def crear_disco_prueba(ruta_base):
    """
    Crea la estructura de carpetas y archivos vacios en ruta_base.
    Los archivos son vacios — solo sirven para probar el normalizado.
    """
    carpetas = [
        "PROGRAMAS/BARRIO ADENTRO",
        "PROGRAMAS/EN LA RAYA",
        "PROGRAMAS/DEJAME SER ARTE/PROGRAMAS",
        "PROGRAMAS/DEJAME SER ARTE/MUSICA",
        "PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA",
        "PROGRAMAS/A PRENDER LA ONDA",
        "PROGRAMAS/PAGANO BOLERO",
        "PROGRAMAS/DIANA URIBE",
        "PROGRAMAS/ENTRE NOSOTROS",
    ]

    archivos = [
        "PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 06-06-25.mp3",
        "PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 17-07-25.mp3",
        "PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES- AGOSTO 25 DE 2025.mp3",
        "PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES DIC 1 DE 2025.mp3",
        "PROGRAMAS/EN LA RAYA/En la Raya Deportes- Septiembre 29 de 2025.mp3",
        "PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DEJAME SER ARTE 03-07-25.mp3",
        "PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DSA 02-10-25.mp3",
        "PROGRAMAS/DEJAME SER ARTE/MUSICA/Arcangel - Andan Diciendo.mp3",
        "PROGRAMAS/DEJAME SER ARTE/MUSICA/GAMBOA - Fuiste Tu.mp3",
        "PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/CABEZOTE DSA.mp3",
        "PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/PISADOR DSA 1.mp3",
        "PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/SEPARADOR DSA 3 FINAL.mp3",
        "PROGRAMAS/A PRENDER LA ONDA/Episodio 35 2025.mp3",
        "PROGRAMAS/A PRENDER LA ONDA/Episodio 118.mp3",
        "PROGRAMAS/A PRENDER LA ONDA/Episodio 122 2023 (1).mp3",
        "PROGRAMAS/A PRENDER LA ONDA/Episodio 290 2025.mp3",
        "PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - mujeres bolero parte 2.mp3",
        "PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - boleros inspirados estaciones.mp3",
        "PROGRAMAS/DIANA URIBE/1 - La historia de la radio y la radio en la historia.mp3",
        "PROGRAMAS/DIANA URIBE/7 - Comienzos de la radio en Colombia.mp3",
        "PROGRAMAS/DIANA URIBE/13 - Radio educativa, Radio Sutatenza.mp3",
        "PROGRAMAS/ENTRE NOSOTROS/PROG. 293 ENTRE NOSOTROS, SABADO 14 DE MARZO 2026.mp3",
        "PROGRAMAS/ENTRE NOSOTROS/PROGRAMA 286 ENTRE NOSOTROS DE KAREN A.mp3",
    ]

    base = Path(ruta_base)

    # Crear carpetas
    for carpeta in carpetas:
        (base / carpeta).mkdir(parents=True, exist_ok=True)

    # Crear archivos vacios
    for archivo in archivos:
        ruta_archivo = base / archivo
        ruta_archivo.parent.mkdir(parents=True, exist_ok=True)
        ruta_archivo.touch()   # crea el archivo vacio

    print(f"Carpetas creadas : {len(carpetas)}")
    print(f"Archivos creados : {len(archivos)}")
    print(f"Ruta             : {base.resolve()}")
    return base

### Ejecutar: crear la estructura en disco

In [ ]:
# Cambiar la ruta si quieres crearlo en otro lugar
RUTA_PRUEBA = "disco_prueba"

disco = crear_disco_prueba(RUTA_PRUEBA)
print()
print("Estructura creada:")
for ruta in sorted(disco.rglob("*")):
    nivel = len(ruta.relative_to(disco).parts) - 1
    prefijo = "  " * nivel + ("| " if ruta.is_file() else "+ ")
    print(prefijo + ruta.name)

Carpetas creadas : 9
Archivos creados : 23
Ruta             : C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\ponencia\Python_Cursos\Notebooks\disco_prueba

Estructura creada:
+ PROGRAMAS
  + A PRENDER LA ONDA
    | Episodio 118.mp3
    | Episodio 122 2023 (1).mp3
    | Episodio 290 2025.mp3
    | Episodio 35 2025.mp3
    | episodio_2023_ep122.mp3
    | episodio_2025_ep035.mp3
    | episodio_2025_ep290.mp3
    | episodio_sin_fecha_ep118.mp3
  + BARRIO ADENTRO
    | BARRIO ADENTRO 06-06-25.mp3
    | BARRIO ADENTRO 17-07-25.mp3
    | barrio_adentro_06_06_25_2025-06-06.mp3
    | barrio_adentro_07_25_2025-07-17.mp3
  + DEJAME SER ARTE
    + MUSICA
      | Arcangel - Andan Diciendo.mp3
      | arcangel_andan_diciendo_sin_fecha.mp3
      | GAMBOA - Fuiste Tu.mp3
      | gamboa_fuiste_tu_sin_fecha.mp3
    + PRODUCCION DSA
      | CABEZOTE DSA.mp3
      | cabezote_dsa_sin_fecha.mp3
      | PISADOR DSA 1.mp3
      | pisador_dsa_sin_fecha.mp3
      | SEPARADOR DSA 3 FINAL.mp3
      |

### Escanear los archivos creados

In [ ]:
# Escanear el disco de prueba que acabamos de crear
# (rglob busca todos los .mp3 dentro de todas las subcarpetas)
archivos = []
for ruta in Path(RUTA_PRUEBA).rglob("*.mp3"):
    archivos.append(ruta.name)   # solo el nombre, sin la ruta

print(f"Archivos encontrados: {len(archivos)}")
print()
for nombre in archivos:
    print(" ", nombre)

Archivos encontrados: 46

  Episodio 118.mp3
  Episodio 122 2023 (1).mp3
  Episodio 290 2025.mp3
  Episodio 35 2025.mp3
  episodio_2023_ep122.mp3
  episodio_2025_ep035.mp3
  episodio_2025_ep290.mp3
  episodio_sin_fecha_ep118.mp3
  BARRIO ADENTRO 06-06-25.mp3
  BARRIO ADENTRO 17-07-25.mp3
  barrio_adentro_06_06_25_2025-06-06.mp3
  barrio_adentro_07_25_2025-07-17.mp3
  Arcangel - Andan Diciendo.mp3
  arcangel_andan_diciendo_sin_fecha.mp3
  GAMBOA - Fuiste Tu.mp3
  gamboa_fuiste_tu_sin_fecha.mp3
  CABEZOTE DSA.mp3
  cabezote_dsa_sin_fecha.mp3
  PISADOR DSA 1.mp3
  pisador_dsa_sin_fecha.mp3
  SEPARADOR DSA 3 FINAL.mp3
  separador_dsa_final_sin_fecha.mp3
  DEJAME SER ARTE 03-07-25.mp3
  dejame_ser_arte_03_07_25_2025-07-03.mp3
  DSA 02-10-25.mp3
  dsa_02_25_2025-10-02.mp3
  1 - La historia de la radio y la radio en la historia.mp3
  13 - Radio educativa, Radio Sutatenza.mp3
  7 - Comienzos de la radio en Colombia.mp3
  comienzos_radio_en_colombia_sin_fecha_ep007.mp3
  historia_radio_y_radio_

### Normalizar todos los archivos encontrados

In [ ]:
# Normalizar todos los archivos encontrados y mostrar el resultado
print(f"{'ANTES':<60} {'DESPUES'}")
print("-" * 120)

for nombre in archivos:
    nuevo = normalizar_nombre(nombre)
    print(f"{nombre:<60} {nuevo}")

ANTES                                                        DESPUES
------------------------------------------------------------------------------------------------------------------------
Episodio 118.mp3                                             episodio_sin_fecha_ep118.mp3
Episodio 122 2023 (1).mp3                                    episodio_2023_ep122.mp3
Episodio 290 2025.mp3                                        episodio_2025_ep290.mp3
Episodio 35 2025.mp3                                         episodio_2025_ep035.mp3
episodio_2023_ep122.mp3                                      episodio_2023_ep122_sin_fecha.mp3
episodio_2025_ep035.mp3                                      episodio_2025_ep035_sin_fecha.mp3
episodio_2025_ep290.mp3                                      episodio_2025_ep290_sin_fecha.mp3
episodio_sin_fecha_ep118.mp3                                 episodio_sin_fecha_ep118_sin_fecha.mp3
BARRIO ADENTRO 06-06-25.mp3                                  barrio_adentro_06_0

### Renombrar los archivos en disco
> Ejecuta esto solo cuando estes seguro del resultado de la tabla anterior.

In [ ]:
# Renombrar los archivos reales en disco
# Revisa la tabla de arriba antes de ejecutar esto.

renombrados = 0
for ruta in Path(RUTA_PRUEBA).rglob("*.mp3"):
    nombre_nuevo = normalizar_nombre(ruta.name)
    ruta_nueva   = ruta.parent / nombre_nuevo

    if ruta.name != nombre_nuevo:               # solo si cambia algo
        ruta.rename(ruta_nueva)
        print(f"  {ruta.name}")
        print(f"  -> {nombre_nuevo}")
        print()
        renombrados += 1

print(f"Total renombrados: {renombrados}")

FileExistsError: [WinError 183] No se puede crear un archivo que ya existe: 'disco_prueba\\PROGRAMAS\\A PRENDER LA ONDA\\Episodio 118.mp3' -> 'disco_prueba\\PROGRAMAS\\A PRENDER LA ONDA\\episodio_sin_fecha_ep118.mp3'